In [ ]:
import xml.etree.ElementTree as ET
import pyspark.sql.functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

def transformar_xml_entsoe(raw_table_name, pais_nombre):
    raw_df = spark.read.table(raw_table_name).collect()
    xml_content = raw_df[0]["raw_xml"]
    root = ET.fromstring(xml_content)
    
    ns = {'ns': 'urn:entsoe.eu:wgedi:components'}
    registros = []
    
    for ts in root.findall('.//ns:TimeSeries', ns):
        start_str = ts.find('.//ns:start', ns).text # Formato: 2026-06-07T22:00Z
        start_dt = datetime.strptime(start_str, "%Y-%m-%dT%H:%MZ")
        
        for point in ts.findall('.//ns:Point', ns):
            pos = int(point.find('ns:position', ns).text)
            precio = float(point.find('ns:price.amount', ns).text)
            # Incrementos de 15 minutos (PT15M)
            timestamp_registro = start_dt + timedelta(minutes=(pos - 1) * 15)
            registros.append((timestamp_registro.strftime("%Y-%m-%d %H:%M:%S"), precio, pais_nombre, "ENTSO-E"))
            
    schema = StructType([
        StructField("timestamp_utc", StringType(), True),
        StructField("precio_mwh", DoubleType(), True),
        StructField("pais", StringType(), True),
        StructField("fuente", StringType(), True)
    ])
    return spark.createDataFrame(registros, schema)

In [ ]:
print("🧹 Procesando y limpiando datos para la Capa Silver...")

# 1. Transformar España y Rumanía desde XML
df_es_silver = transformar_xml_entsoe("bronze_raw_espana", "España")
df_ro_silver = transformar_xml_entsoe("bronze_raw_rumania", "Rumania")

# 2. Transformar Alemania (JSON)
df_de_raw = spark.read.table("bronze_raw_alemania")
# Nota: La lógica interna de desanidación de SMARD se aplica aquí expandiendo los arrays
# Para fines de síntesis, asumimos el dataframe tipado resultante:
df_de_silver = df_de_raw.select(
    F.to_timestamp(F.col("timestamp_utc")).alias("timestamp_utc"),
    F.col("precio").cast("double").alias("precio_mwh"),
    F.lit("Alemania").alias("pais"),
    F.lit("SMARD").alias("fuente")
)

# 3. Transformar Polonia (JSON - Manteniendo moneda de origen PLN)
df_pl_silver = spark.read.table("bronze_raw_polonia").select(
    F.to_timestamp(F.col("dtime_utc")).alias("timestamp_utc"),
    F.col("rce_pln").cast("double").alias("precio_mwh"),
    F.lit("Polonia").alias("pais"),
    F.lit("PSE").alias("fuente")
)

# Aplicar Control de Duplicados Estratégico (Garantiza Idempotencia)
def aplicar_calidad_silver(df):
    return df.dropDuplicates(["timestamp_utc", "pais"]) \
             .withColumn("timestamp_utc", F.col("timestamp_utc").cast("string"))

# Guardar en tablas Silver optimizadas en formato Delta
aplicar_calidad_silver(df_es_silver).write.format("delta").mode("overwrite").saveAsTable("silver_precios_espana")
aplicar_calidad_silver(df_ro_silver).write.format("delta").mode("overwrite").saveAsTable("silver_precios_rumania")
aplicar_calidad_silver(df_de_silver).write.format("delta").mode("overwrite").saveAsTable("silver_precios_alemania")
aplicar_calidad_silver(df_pl_silver).write.format("delta").mode("overwrite").saveAsTable("silver_precios_polonia")

print("✔️ Tablas Silver creadas e indexadas bajo Delta Lake.")